[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/main/examples/entity_alignment.ipynb)

# Align two product cards

Two catalogues describe the same kettle in different words, or two different kettles that share a brand. "Are these the same product?" is a score. Two yes/no questions ask whether the brand matches and whether the model number matches. Code trusts the score: at or above 2.5 the pair is the same item, below 1.5 it is not, and the band in between waits for a person. The yes/no answers are printed next to the score and are not a veto, because a paraphrase can move them on its own.

The cards are plain sentences. The same facts as a JSON object stayed near the middle of the scale even when the model number matched.

Cards are synthetic.


In [ ]:
%pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0"


In [1]:
import os
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


frida-decisions 0.4.0, loaded on cpu, torch.float32


`score` is an expected level from 0 to 4. On these prose cards a real match landed near 3.2 and a different model near 0.2, so 2.5 and 1.5 leave a band for review.


In [2]:
DATA = [
    {"gold": "same", "text": "Карточка А: электрочайник Philips, артикул HD9350, объём 1.7 литра. Карточка Б: чайник Philips модели HD9350 на 1.7 литра."},
    {"gold": "same", "text": "Карточка А: наушники Sony, артикул WH-1000XM5, цвет чёрный. Карточка Б: беспроводные наушники Sony WH-1000XM5, чёрные."},
    {"gold": "different", "text": "Карточка А: электрочайник Philips, артикул HD9350, объём 1.7 литра. Карточка Б: электрочайник Philips, артикул HD4649, объём 1.5 литра."},
    {"gold": "different", "text": "Карточка А: молоко «Домик в деревне» 3.2%, 1 литр. Карточка Б: стол IKEA Linnmon, белый, ширина 100 см."},
    {"gold": "different", "text": "Карточка А: стул IKEA Adde, белый. Карточка Б: стол IKEA Linnmon, белый, ширина 100 см."},
    {"gold": "different", "text": "Карточка А: чай Greenfield Golden Ceylon, 100 пакетиков. Карточка Б: чай Greenfield Flying Dragon, 100 пакетиков."},
]
SAME = [
    "Точно разные товары.",
    "Скорее разные товары.",
    "Неясно, тот же товар или другой.",
    "Скорее один и тот же товар.",
    "Точно один и тот же товар.",
]
SAME_MIN = 2.5
DIFFERENT_MAX = 1.5


def request_for(text):
    return {
        "state": text,
        "questions": {
            "same": {"type": "score", "instructions": "Это один и тот же товар в двух карточках?",
                     "criteria": SAME},
            "brand": {"type": "noul", "instructions": "Бренд в двух карточках один и тот же?"},
            "model": {"type": "noul", "instructions": "Артикул или модель в двух карточках совпадают?"},
        },
    }


def decide(answers):
    score = answers["same"]["score"]
    if score >= SAME_MIN:
        outcome = "same"
    elif score < DIFFERENT_MAX:
        outcome = "different"
    else:
        outcome = "review"
    return {
        "outcome": outcome,
        "score": score,
        "brand": answers["brand"]["noul"],
        "model": answers["model"]["noul"],
    }


The yes/no questions are printed beside the score. They are not a veto.


In [3]:
def run(judge):
    hits = 0
    decided = 0
    for row in DATA:
        decision = decide(judge(request_for(row["text"]))["answers"])
        if decision["outcome"] == row["gold"]:
            mark = "ok"
            hits += 1
        else:
            mark = "miss"
        if decision["outcome"] != "review":
            decided += 1
        print(f"{mark:4} got {decision['outcome']:10} gold {row['gold']:10} "
              f"score {decision['score']:.2f}  brand {decision['brand']:.2f}  model {decision['model']:.2f}")
        print(f"     {row['text']}")
    print(f"\n{hits}/{len(DATA)} outcomes match the label, {decided}/{len(DATA)} decided without review")


In [4]:
run(judge)


ok   got same       gold same       score 3.50  brand 0.99  model 0.99
     Карточка А: электрочайник Philips, артикул HD9350, объём 1.7 литра. Карточка Б: чайник Philips модели HD9350 на 1.7 литра.
ok   got same       gold same       score 3.31  brand 0.93  model 0.94
     Карточка А: наушники Sony, артикул WH-1000XM5, цвет чёрный. Карточка Б: беспроводные наушники Sony WH-1000XM5, чёрные.
ok   got different  gold different  score 1.00  brand 0.35  model 0.27
     Карточка А: электрочайник Philips, артикул HD9350, объём 1.7 литра. Карточка Б: электрочайник Philips, артикул HD4649, объём 1.5 литра.
ok   got different  gold different  score 0.93  brand 0.01  model 0.00
     Карточка А: молоко «Домик в деревне» 3.2%, 1 литр. Карточка Б: стол IKEA Linnmon, белый, ширина 100 см.
ok   got different  gold different  score 1.12  brand 0.29  model 0.20
     Карточка А: стул IKEA Adde, белый. Карточка Б: стол IKEA Linnmon, белый, ширина 100 см.
ok   got different  gold different  score 0.90  br